In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
ascending = True
k = 10
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- to_rank ---
FIX_TO_RANK_PRIMARY_KEY = pd.Series([1,2,3], name="id")
FIX_TO_RANK_SCORE = pd.Series([0.8,0.5,0.3], name="score")
FIX_TO_RANK_PRIMARY_KEY_PL = pl.Series("id", [1,2,3])
FIX_TO_RANK_SCORE_PL = pl.Series("score", [0.8,0.5,0.3])
FIX_TO_RANK_PRIMARY_KEY_PL = pl.Series("id", [1,2,3])
FIX_TO_RANK_SCORE_PL = pl.Series("score", [0.8,0.5,0.3])

print("✅ Fixtures loaded")

✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_to_rank(primary_key, score):
    df = pd.DataFrame({primary_key.name: primary_key, score.name: score}).set_index(
        primary_key.name, drop=True
    )
    df = df.sort_values(by=str(score.name), ascending=ascending)
    df["rank"] = np.ceil(np.arange(1, len(df) + 1) / len(df) * k).astype(int)
    return df

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_to_rank(primary_key, score):
    import numpy as np

    df = pl.DataFrame({primary_key.name: primary_key, score.name: score}).drop(primary_key.name)
    df = df.sort(by=str(score.name), descending=not ascending)
    if df.height == 0:
        df = df.with_columns(pl.Series("rank", [], dtype=pl.Int64))
    else:
        df = df.with_columns(
            (
                (pl.int_range(1, pl.len() + 1, eager=False) / pl.len() * k)
                .ceil()
                .cast(pl.Int64)
            ).alias("rank")
        )
    return df

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: to_rank ===

# L1 smoke – generated
try:
    _r = gen_to_rank(FIX_TO_RANK_PRIMARY_KEY_PL, FIX_TO_RANK_SCORE_PL)
    print("✅ L1 smoke gen_to_rank: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_to_rank: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_to_rank(FIX_TO_RANK_PRIMARY_KEY, FIX_TO_RANK_SCORE)
    print("✅ L1 smoke before_to_rank: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_to_rank: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_to_rank(FIX_TO_RANK_PRIMARY_KEY, FIX_TO_RANK_SCORE)
    _rg = gen_to_rank(FIX_TO_RANK_PRIMARY_KEY_PL, FIX_TO_RANK_SCORE_PL)
    compare(_rb, _rg, "to_rank", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence to_rank: setup error — {type(_e).__name__}: {_e}")

# L3 edge – schema-bearing empty Series on both sides
try:
    _rb = before_to_rank(
        pd.Series([], name="id", dtype="int64"),
        pd.Series([], name="score", dtype="float64"),
    )
    _rg = gen_to_rank(
        pl.Series("id", [], dtype=pl.Int64),
        pl.Series("score", [], dtype=pl.Float64),
    )
    compare(_rb, _rg, "L3 edge to_rank empty Series", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge to_rank empty Series: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_to_rank: OK, type= DataFrame
✅ L1 smoke before_to_rank: OK
❌ L2 equivalence to_rank: MISMATCH — column sets differ (before-only={'id'}, gen-only=set())
❌ L3 edge to_rank empty Series: MISMATCH — column sets differ (before-only={'id'}, gen-only=set())
